In [ ]:
# XG Boost
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# 1. Define your LightGBM best parameters
lgbm_best = {
    "colsample_bytree": 0.49843120969470817,
    "learning_rate": 0.021523244311830798,
    "max_depth": 8,
    "min_child_samples": 450,
    "n_estimators": 780,
    "num_leaves": 88,
    "reg_alpha": 0.03572601757164547,
    "reg_lambda": 0.023836232805870886
}

# 2. Map and translate safely to XGBoost properties
xgb_params = {
    "n_estimators": int(lgbm_best["n_estimators"]),
    "max_depth": int(lgbm_best["max_depth"]),
    "learning_rate": lgbm_best["learning_rate"],
    "colsample_bytree": lgbm_best["colsample_bytree"],
    "reg_alpha": lgbm_best["reg_alpha"],
    "reg_lambda": lgbm_best["reg_lambda"],
    
    # XGBoost structural equivalents
    "min_child_weight": 5, 
    
    # Hardware & Data optimizations
    "device": "cuda",                  # Runs on your NVIDIA GPU via CUDA 12.5
    "enable_categorical": True,        # Natively reads your .astype('category') columns
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "random_state": 1994,
    "n_jobs": -1
}
# 3. Initialize the XGBoost Classifier
xgb_model = xgb.XGBClassifier(**xgb_params)

# 4. Fit the model using early stopping
print("Fitting your final XGBoost model on GPU...")
xgb_model.fit(
    X_train_final, y_train_final,
    eval_set=[(X_val_final, y_val_final)],
    verbose=50,  # Prints out the AUC progress every 50 trees
    )
# Evaluate on test set
y_pred_proba = xgb_model.predict_proba(X_val_final)[:, 1]
test_auc = roc_auc_score(y_val_final, y_pred_proba)
print(f"Test AUC: {test_auc:.4f}")
# 5. Check your final evaluation score
val_preds = xgb_model.predict_proba(X_val_final)[:, 1]
final_auc = roc_auc_score(y_val_final, val_preds)
print(f"\nTraining Complete! Final Validation ROC-AUC: {final_auc:.6f}")
# --- 4. Find the Optimal Threshold from the Precision-Recall Curve ---
# Get precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_val_final, y_pred_proba)

# Calculate F1-score for each threshold
# Note: We add a small epsilon to avoid division by zero
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

# Find the threshold that maximizes the F1-score
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_f1_idx]
best_f1_score = f1_scores[best_f1_idx]

print(f"\n--- Finding the Optimal Threshold ---")
print(f"Best Threshold found: {best_threshold:.4f}")
print(f"F1-Score at this threshold: {best_f1_score:.4f}\n")

# Evaluate performance at the new, optimal threshold
y_pred_optimal = (y_pred_proba >= best_threshold) * 1

print(f"--- Performance at Optimal Threshold ({best_threshold:.2f}) ---")
print(classification_report(y_val_final, y_pred_optimal))
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
# Handle missing values
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.20, random_state=17,stratify=y)

# Fit logistic regression model
model = LogisticRegression(random_state=124, max_iter=10000, penalty='l1', solver='liblinear', C=0.1)
model.fit(X_train, y_train)

# Create DataFrame with coefficients
coef_df = pd.DataFrame({'variable': feature_cols, 'coef': model.coef_[0]})
y_pred_proba = model.predict_proba(X_val)[:, 1]  # This will give the predicted probabilities for the validation set
# --- 4. Find the Optimal Threshold from the Precision-Recall Curve ---
# Get precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_val, y_pred_proba)

# Calculate F1-score for each threshold
# Note: We add a small epsilon to avoid division by zero
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

# Find the threshold that maximizes the F1-score
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_f1_idx]
best_f1_score = f1_scores[best_f1_idx]

print(f"\n--- Finding the Optimal Threshold ---")
print(f"Best Threshold found: {best_threshold:.4f}")
print(f"F1-Score at this threshold: {best_f1_score:.4f}\n")

# Evaluate performance at the new, optimal threshold
y_pred_optimal = (y_pred_proba >= best_threshold) * 1

print(f"--- Performance at Optimal Threshold ({best_threshold:.2f}) ---")
print(classification_report(y_val, y_pred_optimal))
roc_auc = roc_auc_score(y_val, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")
# --- 1. Define the Objective Function ---
def objective(params):
    """
    The objective function for Hyperopt to minimize.
    It trains a LightGBM model and returns the validation loss.
    """
    # Hyperopt sends integer parameters as floats, so we need to convert them
    params['num_leaves'] = int(params['num_leaves'])
    params['max_depth'] = int(params['max_depth'])
    params['n_estimators'] = int(params['n_estimators'])
    
    # Initialize the XGBoost model
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        random_state=42,
        use_label_encoder=False,
        n_jobs=-1,
        **params  # Unpack the dictionary of hyperparameters
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    mean_auc = cross_val_score(model, X_train, y_train,
                               scoring='roc_auc', cv=cv, n_jobs=-1).mean()

    # Return the standard Hyperopt dictionary format
    return {'loss': -mean_auc, 'status': STATUS_OK}
# --- 3. Run the Optimization ---
# The Trials object will store the history of the search
trials = Trials()

# fmin is the optimization function
best_params = fmin(
    fn=objective,          # The function to minimize
    space=space,           # The hyperparameter space
    algo=tpe.suggest,      # The search algorithm (Tree-of-Parzen-Estimators)
    max_evals=150,         # The number of trials to run
    trials=trials          # The object to store the results
)

print("\n--- Hyperopt Search Complete ---")
print("Best hyperparameters found:")
# Convert float values back to int for integer parameters
best_params['n_estimators'] = int(best_params['n_estimators'])
best_params['num_leaves'] = int(best_params['num_leaves'])
best_params['max_depth'] = int(best_params['max_depth'])
print(best_params)
best_params  = {
    'colsample_bytree': np.float64(0.8714610650925534), 
    'max_depth': 3, 
    'n_estimators': 50, 
    'num_leaves': 57, 
    'reg_alpha': np.float64(0.4749466419949838), 
    'reg_lambda': np.float64(0.14089776932425035)
    }
# 1. Initialize the XGBoost model
# 'eval_metric' is set to avoid a future deprecation warning.
xgboost_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='auc',
    **best_params
)

# 2. Train the model
print("Training XGBoost...")
xgboost_model.fit(X_train, y_train)

# 3. Make predictions and evaluate
y_pred_proba = xgboost_model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)

print(f"\nXGBoost AUC Score: {auc:.4f}")
# --- 4. Find the Optimal Threshold from the Precision-Recall Curve ---
# Get precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_val, y_pred_proba)

# Calculate F1-score for each threshold
# Note: We add a small epsilon to avoid division by zero
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

# Find the threshold that maximizes the F1-score
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_f1_idx]
best_f1_score = f1_scores[best_f1_idx]

print(f"\n--- Finding the Optimal Threshold ---")
print(f"Best Threshold found: {best_threshold:.4f}")
print(f"F1-Score at this threshold: {best_f1_score:.4f}\n")

# Evaluate performance at the new, optimal threshold
y_pred_optimal = (y_pred_proba >= best_threshold) * 1

print(f"--- Performance at Optimal Threshold ({best_threshold:.2f}) ---")
print(classification_report(y_val, y_pred_optimal))